# Student Performance Analysis & Prediction System
**MissAlpha™ AI Labs | Day 1–Project Implementation**

This notebook covers environment verification, data loading, data quality checks, exploratory analysis, visualisations, and a basic machine-learning comparison.

**Learning objective:** understand the dataset before modelling. Do not treat model output as a definitive judgement about a student.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Locate and load the dataset
The CSV is stored in `data/student_data.csv`. The loader also handles semicolon-separated CSV files.

In [ ]:
DATA_PATH = Path("data") / "student_data.csv"
if not DATA_PATH.exists():
    # Also supports running this notebook from the project root after moving the CSV beside it.
    DATA_PATH = Path("student_data.csv")

try:
    df = pd.read_csv(DATA_PATH)
    if df.shape[1] == 1:
        df = pd.read_csv(DATA_PATH, sep=";")
except Exception:
    df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

## 3. Preview the first 10 records

In [ ]:
df.head(10)

## 4. Dataset dimensions and column names

In [ ]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("\nColumn names:")
for number, column in enumerate(df.columns, start=1):
    print(number, column)

## 5. Data types and structure

In [ ]:
df.info()

## 6. Missing values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0] if (missing > 0).any() else "No missing values found."

## 7. Duplicate records

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate records:", duplicate_count)

## 8. Descriptive statistical summary

In [ ]:
df.describe(include='all').transpose()

## 9. Clean and prepare target
For the common student performance dataset, `G3` is the final grade. If the supplied file uses a different schema, the last column is used as a fallback; verify it before modelling.

In [ ]:
target = "G3" if "G3" in df.columns else df.columns[-1]
df[target] = pd.to_numeric(df[target], errors="coerce")
analysis_df = df.dropna(subset=[target]).copy()
print("Target column:", target)
print("Rows available for analysis:", len(analysis_df))

## 10. Explore the target distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=analysis_df, x=target, bins=15, kde=True)
plt.title(f"Distribution of {target}")
plt.xlabel(target)
plt.tight_layout()
plt.show()

## 11. Explore study time and performance (when columns exist)

In [ ]:
if "studytime" in analysis_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=analysis_df, x="studytime", y=target)
    plt.title(f"{target} by study-time category")
    plt.tight_layout()
    plt.show()
else:
    print("No studytime column in this dataset.")

## 12. Explore absences and performance (when columns exist)

In [ ]:
if "absences" in analysis_df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=analysis_df, x="absences", y=target, alpha=0.7)
    plt.title(f"{target} vs absences")
    plt.tight_layout()
    plt.show()
else:
    print("No absences column in this dataset.")

## 13. Numerical correlation

In [ ]:
numeric_df = analysis_df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(11, 8))
    sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
    plt.title("Correlation between numerical variables")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for a correlation heatmap.")

## 14. Prepare a prediction model

This is a regression task because the target is a numeric grade. The example below excludes `G1` and `G2` when predicting `G3`, so it demonstrates an early-warning model based on non-grade factors. A real deployment must use a clearly defined prediction time and only information available at that time.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

excluded = {target}
if target == "G3":
    excluded.update(c for c in ["G1", "G2"] if c in analysis_df.columns)

X = analysis_df.drop(columns=list(excluded), errors="ignore")
y = analysis_df[target]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 15. Train and compare baseline and regression models

In [ ]:
estimators = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, min_samples_leaf=2
    )
}

trained_models = {}
scores = []

for model_name, estimator in estimators.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    model_pipeline.fit(X_train, y_train)
    predictions = model_pipeline.predict(X_test)
    trained_models[model_name] = model_pipeline
    scores.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

model_results = pd.DataFrame(scores).sort_values("MAE")
model_results

## 16. Interpret evaluation metrics
- **MAE:** average absolute prediction error, in grade points; lower is better.
- **RMSE:** penalises larger errors more strongly; lower is better.
- **R²:** compares model fit with predicting the mean; closer to 1 is generally better, but can be negative.

Do not select a model using one metric alone. Review the test design, error patterns, and fairness considerations.

In [ ]:
best_model_name = model_results.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best model by test MAE:", best_model_name)

## 17. Save results and model

In [ ]:
import joblib

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
model_results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
joblib.dump({
    "pipeline": best_model,
    "features": X.columns.tolist(),
    "target": target,
    "model_name": best_model_name
}, OUTPUT_DIR / "student_performance_model.joblib")

df.describe(include="all").transpose().to_csv(
    OUTPUT_DIR / "descriptive_statistics.csv"
)
print("Saved model and results in:", OUTPUT_DIR.resolve())

## 18. Key findings and conclusion

Complete this section after running the notebook:
1. Describe the dataset size and the most important columns.
2. State whether missing values or duplicate records were found.
3. Describe at least two patterns visible in the charts.
4. Report the test-set MAE, RMSE and R² for the models.
5. Explain one limitation and one improvement.

**Responsible-use note:** grades can be influenced by many factors not captured in the dataset. Do not use a prediction as the sole basis for student support, discipline, admissions, or other high-impact decisions.